In [26]:
import math
import json
import ast
import string
import re
from pathlib import Path


import pandas as pd
from dotenv import load_dotenv

load_dotenv()

from vbree.providers.hf_provider import HfProvider
from vbree.utils.parse import extract_json, validate_letter

In [27]:
RUNS_DIR = Path("../runs")
print(RUNS_DIR)
#TEMPERATURE = 0.0

..\runs


In [28]:
df = pd.read_csv(RUNS_DIR/ "mmlu_pro_sample.csv")

# if choices were saved as strings in CSV, convert them back to Python lists
def parse_choices(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

df["options"] = df["options"].apply(parse_choices)


print(df.shape)
df.head()

(3500, 8)


,question_id,question,options,answer,answer_index,cot_content,category,src
0,2926,"Stomata allow gases, such as H2O and CO2, to e...",[CO2 diffuses out of the leaf faster than H2O ...,I,8,NaN,biology,ori_mmlu-high_school_biology
1,3139,Even though sex determination does not appear ...,[Sex determination in somedioecious organisms ...,I,8,NaN,biology,stemez-Genetics
2,2845,"Why are the elements carbon, hydrogen, oxygen,...",[These elements are the only elements that can...,B,1,NaN,biology,stemez-Biology
3,3445,The ability of the brain to detect differences...,"[The number of synapses crossed, The speed of ...",F,5,NaN,biology,ori_mmlu-college_biology
4,3323,Which of the following statements is correct?,[Smaller DNA fragments migrate more rapidly th...,A,0,NaN,biology,ori_mmlu-high_school_biology


In [29]:
def build_single_agent_prompt(question: str, choices: list[str]) -> str:
    letters = string.ascii_uppercase[:len(choices)]
    choices_str = "\n".join([f"{letters[i]}. {choice}" for i, choice in enumerate(choices)])
    allowed_letters = ", ".join(letters)

    return f"""You are an expert answering a multiple choice question.

Question: {question}

Possible Choices:
{choices_str}


=== STEP 1 — THINK FREELY ===
Reason through this problem carefully.
- Analyze each choice and explain your logic
- No word limit here — maximum 900 tokens total


=== STEP 2 — OUTPUT JSON ===
After your reasoning above, output ONLY this JSON on a single line.
- Return ONLY valid JSON on a SINGLE LINE
- No newlines, no markdown, no bullet points, no LaTeX
- No newlines inside the response field
- No markdown formatting like ** or *
- No bullet points or numbered lists in the response
- Start with {{ and end with }}

The "letter" field must contain the letter of YOUR chosen answer from the possible choices above.

Target Schema:
{{"letter": "<one of: {allowed_letters}>"}}

Example Output:
{{"letter": "C"}}
"""

In [30]:
def parse_single_agent_output(raw: str, n_choices: int) -> tuple[str, str]:
    obj = extract_json(raw)
    letter = validate_letter(obj.get("letter", ""), n_choices)
    response = str(obj.get("response", "")).strip()

    if not response and raw:
        response = str(raw).strip()

    return letter, response

In [31]:
def run_single_agent_batch(
    data: pd.DataFrame,
    provider,
    model_name: str,
    batch_size: int = 500,
    output_dir: Path = Path("../runs/single_agent_batches")
):
    output_dir.mkdir(parents=True, exist_ok=True)

    n_rows = len(data)
    n_batches = math.ceil(n_rows / batch_size)

    for batch_idx in range(n_batches):
        start = batch_idx * batch_size
        end = min((batch_idx + 1) * batch_size, n_rows)

        batch_file = output_dir / f"single_{model_name}_batch_{batch_idx:03d}.csv"

        # skip already completed batch
        if batch_file.exists():
            print(f"Skipping batch {batch_idx} (already exists)")
            continue

        batch_df = data.iloc[start:end].copy()
        rows = []

        print(f"Running batch {batch_idx+1}/{n_batches} | rows {start} to {end-1}")

        for i, row in batch_df.iterrows():
            try:
                prompt = build_single_agent_prompt(
                    question=row["question"],
                    choices=row["options"]
                )
                
                raw = provider.generate(prompt=prompt)
                letter, response = parse_single_agent_output(raw, n_choices=len(row["options"]))

                rows.append({
                    "question_id": row["question_id"],
                    "question": row["question"],
                    "choices": row["options"],
                    "domain": row["category"],
                    "answer": row["answer"],
                    "cot_content": row.get("cot_content", ""),
                    "model": model_name,
                    "selected_choice": letter,
                    "updated_answer": response
                })

                if (start + i) % 250 == 0:
                    print(f"Processed {start + i} / {batch_size} rows")

            except Exception as e:
                rows.append({
                    "question_id": row["question_id"],
                    "question": row["question"],
                    "choices": row["options"],
                    "domain": row["category"],
                    "answer": row["answer"],
                    "cot_content": row.get("cot_content", ""),
                    "model": model_name,
                    "selected_choice": "",
                    "updated_answer": f"ERROR: {str(e)}"
                })

        batch_results = pd.DataFrame(rows)
        batch_results.to_csv(batch_file, index=False)
        print(f"Saved batch to: {batch_file}")


    print("All batches processed.")

In [ ]:
providers = {
    "google/gemma-2-9b-it": HfProvider("google/gemma-2-9b-it", provider="featherless-ai"),
}

single_Mistral = run_single_agent_batch(
    data=df,
    provider=providers["google/gemma-2-9b-it"],
    model_name="gemma"
)

Running batch 1/7 | rows 0 to 499
Processed 0 / 500 rows
Processed 250 / 500 rows
Saved batch to: ..\runs\single_agent_batches\single_Qwen_batch_000.csv
Running batch 2/7 | rows 500 to 999
Processed 1000 / 500 rows
[Qwen/Qwen2.5-7B-Instruct:together] Network error attempt 1/5: [WinError 10054] An existing connection was forcibly closed by the remote host
Waiting 10s before retry...
Processed 1250 / 500 rows
Saved batch to: ..\runs\single_agent_batches\single_Qwen_batch_001.csv
Running batch 3/7 | rows 1000 to 1499
Processed 2000 / 500 rows
Processed 2250 / 500 rows
[Qwen/Qwen2.5-7B-Instruct:together] Network error attempt 1/5: [WinError 10054] An existing connection was forcibly closed by the remote host
Waiting 10s before retry...
[Qwen/Qwen2.5-7B-Instruct:together] generation failed: [WinError 10053] An established connection was aborted by the software in your host machine
[Qwen/Qwen2.5-7B-Instruct:together] Network error attempt 1/5: [Errno 11001] getaddrinfo failed
Waiting 10s bef

In [33]:
def combine_batch_results(model_name: str, batch_dir: Path) -> pd.DataFrame:
    files = sorted(batch_dir.glob(f"single_{model_name}_batch_*.csv"))
    dfs = [pd.read_csv(f) for f in files]
    combined = pd.concat(dfs, ignore_index=True)
    return combined

In [ ]:
BATCH_DIR = Path("../runs/single_agent_batches")
single_gemma_full = combine_batch_results("gemma", BATCH_DIR)
single_gemma_full.to_csv(BATCH_DIR/"single_gemma_full_results.csv", index=False)